In [ ]:
import sys
import os
sys.path.insert(0, os.path.dirname(os.path.abspath("./")))

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import math
import random
import os
import matplotlib.pyplot as plt
from tqdm import tqdm
import numpy as np
import json

from csp import ModNArithmeticGenerator, SymbolicArithmeticDataset
from utils import save_checkpoint, load_checkpoint, setup_logging


# ==================== MiniARFormer 定义 ====================
class LearnablePositionalEncoding(nn.Module):
    def __init__(self, max_len, d_model):
        super().__init__()
        self.pos_embedding = nn.Parameter(torch.randn(1, max_len, d_model) * 0.1)
    
    def forward(self, x):
        seq_len = x.size(1)
        return x + self.pos_embedding[:, :seq_len, :]


def create_causal_mask(seq_len, device):
    mask = torch.triu(torch.ones(seq_len, seq_len, device=device), diagonal=1)
    return mask.bool()


class DecoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn = nn.MultiheadAttention(
            d_model, num_heads, dropout=dropout, batch_first=True
        )
        self.dropout1 = nn.Dropout(dropout)
        self.norm1 = nn.LayerNorm(d_model)
        
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout)
        )
        self.norm2 = nn.LayerNorm(d_model)
    
    def forward(self, x, mask=None):
        attn_out, _ = self.self_attn(x, x, x, attn_mask=mask)
        x = self.norm1(x + self.dropout1(attn_out))
        ffn_out = self.ffn(x)
        x = self.norm2(x + ffn_out)
        return x


class MiniARFormer(nn.Module):
    """
    纯 Decoder 架构，标准自回归训练方式
    训练时：输入 target_ids[:, :-1]，预测 target_ids[:, 1:]
    推理时：从 <SOS> 开始自回归生成
    """
    def __init__(
        self,
        vocab_size,
        d_model=64,
        num_heads=4,
        num_layers=2,
        d_ff=128,
        max_len=256,
        dropout=0.1,
        pad_idx=0,
        sos_idx=1,
        eos_idx=2
    ):
        super().__init__()
        self.vocab_size = vocab_size
        self.d_model = d_model
        self.pad_idx = pad_idx
        self.sos_idx = sos_idx
        self.eos_idx = eos_idx
        self.max_len = max_len
        
        self.token_embedding = nn.Embedding(vocab_size, d_model, padding_idx=pad_idx)
        self.pos_encoding = LearnablePositionalEncoding(max_len, d_model)
        
        self.layers = nn.ModuleList([
            DecoderLayer(d_model, num_heads, d_ff, dropout)
            for _ in range(num_layers)
        ])
        
        self.norm = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size)
        
        self._init_weights()
    
    def _init_weights(self):
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)
    
    def forward(self, input_ids, target_ids=None):
        """
        input_ids: [B, T_in] 输入表达式
        target_ids: [B, T_out] 目标序列（训练时使用）
        """
        window_size = 64  # 滑动窗口大小

        if target_ids is not None:
            # ========== 训练模式 ==========
            B, T_out = target_ids.shape
            if T_out <= 1:
                return torch.zeros(B, 0, self.vocab_size, device=target_ids.device)

            logits_list = []
            current_seq = target_ids[:, :1]  # [B, 1]

            for t in range(T_out - 1):
                # 滑动窗口截断（current_seq 是 [B, T]）
                if current_seq.size(1) > window_size:
                    current_seq = current_seq[:, -window_size:]

                x = self.token_embedding(current_seq)
                x = self.pos_encoding(x)
                mask = create_causal_mask(x.size(1), x.device)

                for layer in self.layers:
                    x = layer(x, mask=mask)
                x = self.norm(x)
                next_logit = self.lm_head(x)[:, -1:, :]  # [B, 1, vocab_size]
                logits_list.append(next_logit)

                # Teacher forcing
                current_seq = torch.cat([current_seq, target_ids[:, t+1:t+2]], dim=1)

            return torch.cat(logits_list, dim=1)

        else:
            # ========== 推理模式 ==========
            generated = input_ids  # [B, T_in]
            input_len = input_ids.size(1)

            for _ in range(self.max_len - generated.size(1)):
                if generated.size(1) > window_size:
                    generated = generated[:, -window_size:]

                x = self.token_embedding(generated)
                x = self.pos_encoding(x)
                mask = create_causal_mask(x.size(1), x.device)

                for layer in self.layers:
                    x = layer(x, mask=mask)
                x = self.norm(x)
                logits = self.lm_head(x)

                next_token = logits[:, -1, :].argmax(dim=-1, keepdim=True)
                generated = torch.cat([generated, next_token], dim=1)

                if (next_token == self.eos_idx).all():
                    break

            # 只返回新生成的部分
            return generated[:, input_len:]


# ==================== 训练与评估函数 ====================
def train_arformer_epoch(model, train_loader, optimizer, criterion, device, pad_idx):
    model.train()
    total_loss = 0
    for batch in tqdm(train_loader, desc='Training'):
        input_ids = batch['input_ids'].to(device)
        output_ids = batch['output_ids'].to(device)
        
        # 标准 Teacher Forcing：用 output_ids[:, :-1] 预测 output_ids[:, 1:]
        logits = model(input_ids, output_ids)  # [B, T_out-1, vocab_size]
        targets = output_ids[:, 1:]  # [B, T_out-1]
        
        # 如果序列长度 <= 1，跳过
        if logits.size(1) == 0:
            continue
        
        # 展平计算 loss
        loss = criterion(
            logits.reshape(-1, logits.size(-1)),
            targets.reshape(-1)
        )
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    
    return total_loss / len(train_loader)


def evaluate_arformer(model, test_loader, device, pad_idx, eos_idx, debug=True):
    model.eval()
    correct = 0
    total = 0
    idx2char = test_loader.dataset.idx2char
    
    with torch.no_grad():
        for batch_idx, batch in enumerate(tqdm(test_loader, desc='Evaluating')):
            input_ids = batch['input_ids'].to(device)
            output_ids = batch['output_ids'].to(device)
            
            # 推理生成
            pred_ids = model(input_ids, target_ids=None)  # [B, T]
            
            # 比较生成的序列和目标序列
            # 注意：需要对齐长度，截取有效部分
            for i in range(len(pred_ids)):
                # 去除 pad 和 eos 后的字符串比较
                pred_tokens = pred_ids[i].tolist()
                target_tokens = output_ids[i].tolist()
                
                # 去除 pad 和 eos
                pred_tokens = [x for x in pred_tokens if x not in [pad_idx, eos_idx]]
                target_tokens = [x for x in target_tokens if x not in [pad_idx, eos_idx]]
                
                pred_str = ''.join(idx2char[x] for x in pred_tokens if x in idx2char)
                target_str = ''.join(idx2char[x] for x in target_tokens if x in idx2char)
                
                if debug and batch_idx == 0 and i < 5:
                    input_tokens = input_ids[i].tolist()
                    input_tokens = [x for x in input_tokens if x not in [pad_idx, eos_idx]]
                    input_str = ''.join(idx2char[x] for x in input_tokens if x in idx2char)
                    print(f"[Debug] Input: {input_str}")
                    print(f"[Debug] Pred : {pred_str}")
                    print(f"[Debug] Target: {target_str}")
                    print(f"pred_ids[0]: {pred_ids[0].tolist()}")
                    print("-" * 40)
                
                if pred_str == target_str:
                    correct += 1
                total += 1
    
    return correct / total


# ==================== 主程序 ====================
def main():
    experiment = 'arformer_teacher_forcing'
    data_mode = 'complete'  # 使用 complete 模式，包含 repeat 和答案
    
    base_dir = os.path.dirname(os.path.abspath("./"))
    model_path = os.path.join(base_dir, experiment)
    fig_path = os.path.join(model_path, 'figure')
    os.makedirs(model_path, exist_ok=True)
    os.makedirs(fig_path, exist_ok=True)
    
    logger = setup_logging()
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")
    
    # ========== 数据 ==========
    gen = ModNArithmeticGenerator(n=9, simple=False)
    
    train_dataset = SymbolicArithmeticDataset(
        200000, max_terms=3, max_digits=1, min_val=0, max_val=9,
        generate_expression_func=gen, vocab=None, mode=data_mode
    )
    test_dataset = SymbolicArithmeticDataset(
        20000, max_terms=3, max_digits=1, min_val=0, max_val=9,
        generate_expression_func=gen, vocab=None, mode=data_mode
    )
    
    train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)
    
    pad_idx = train_dataset.char2idx['<PAD>']
    eos_idx = train_dataset.char2idx['<EOS>']
    sos_idx = train_dataset.char2idx['<SOS>']
    vocab_size = train_dataset.vocab_size
    
    print(f"Vocab size: {vocab_size}, Pad: {pad_idx}, EOS: {eos_idx}, SOS: {sos_idx}")
    
    # ========== 模型 ==========
    model = MiniARFormer(
        vocab_size=vocab_size,
        d_model=64,
        num_heads=4,
        num_layers=2,
        d_ff=128,
        max_len=256,
        pad_idx=pad_idx,
        sos_idx=sos_idx,
        eos_idx=eos_idx
    ).to(device)
    
    total_params = sum(p.numel() for p in model.parameters())
    print(f"参数量: {total_params:,}")
    
    # ========== 优化器 & loss ==========
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
    criterion = nn.CrossEntropyLoss(ignore_index=pad_idx)
    
    # ========== 训练 ==========
    epochs = 50
    cp_path = os.path.join(model_path, f'arformer_{data_mode}.pt')
    best_acc = 0.0
    best_loss = float('inf')
    
    epoch_losses = []
    epoch_accs = []
    
    for epoch in range(epochs):
        loss = train_arformer_epoch(
            model, train_loader, optimizer, criterion, device, pad_idx
        )
        acc = evaluate_arformer(model, test_loader, device, pad_idx, eos_idx, debug=(epoch==0))
        
        epoch_losses.append(loss)
        epoch_accs.append(acc)
        
        print(f"Epoch {epoch+1}: Loss={loss:.4f}, Acc={acc:.4f}")
        
        if acc > best_acc:
            best_acc = acc
            save_checkpoint(model, optimizer, epoch, loss, acc, cp_path)
        
        if loss < best_loss:
            best_loss = loss
    
    print(f"\n最佳准确率: {best_acc:.4f}")
    print(f"模型保存至: {cp_path}")
    
    # ========== 保存结果 ==========
    experiment_results = {
        'model': 'MiniARFormer (Teacher Forcing)',
        'params': total_params,
        'epochs': epochs,
        'best_acc': best_acc,
        'best_loss': best_loss,
        'epoch_losses': epoch_losses,
        'epoch_accs': epoch_accs,
    }
    with open(os.path.join(model_path, 'experiment_results.json'), 'w') as f:
        json.dump(experiment_results, f, indent=2)


if __name__ == "__main__":
    main()

Using device: cuda
Vocab size: 21, Pad: 18, EOS: 17, SOS: 16
参数量: 86,165


Evaluating:   8%|▊         | 24/313 [00:00<00:02, 120.62it/s]

[Debug] Input: <SOS>((8 - (0 + (7 + 2)) - 5)) mod 9 = 
[Debug] Pred : 
[Debug] Target: 3 repeat ((8 - (0 + (7 + 2)) - 5)) mod 9 = 3
pred_ids[0]: []
----------------------------------------
[Debug] Input: <SOS>(((4 + 3) + (5 + 4)) + 4) mod 9 = 
[Debug] Pred : 
[Debug] Target: 2 repeat (((4 + 3) + (5 + 4)) + 4) mod 9 = 2
pred_ids[0]: []
----------------------------------------
[Debug] Input: <SOS>(((0 + 0))) mod 9 = 
[Debug] Pred : 
[Debug] Target: 0 repeat (((0 + 0))) mod 9 = 0
pred_ids[0]: []
----------------------------------------
[Debug] Input: <SOS>((0 + 1) + 1) mod 9 = 
[Debug] Pred : 
[Debug] Target: 2 repeat ((0 + 1) + 1) mod 9 = 2
pred_ids[0]: []
----------------------------------------
[Debug] Input: <SOS>(0 - 0 + (0 - 1)) mod 9 = 
[Debug] Pred : 
[Debug] Target: 8 repeat (0 - 0 + (0 - 1)) mod 9 = 8
pred_ids[0]: []
----------------------------------------


Evaluating: 100%|██████████| 313/313 [00:02<00:00, 144.73it/s]


Epoch 1: Loss=0.5554, Acc=0.0000


Training:  31%|███       | 484/1563 [03:05<06:52,  2.62it/s]